<a href="https://colab.research.google.com/github/jennachateau/ch2-code/blob/main/Ch3_STA_%26_SNOA_Rankings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
from statistics import NormalDist

import numpy as np
import pandas as pd

EXPECTED_UNIVERSE_COUNT = 50
MIN_SECTOR_OBSERVATIONS = 3
DATA_URL = "https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv"
VALIDATION_SNAPSHOT_SHA256 = "b48ba2400340a2447e781abf82333e993aaf45b55ea15ad9138a0e2b196e1813"

df = pd.read_csv(DATA_URL)
full_history_df = df.copy(deep=True)

print("Data loaded from:", DATA_URL)
print("Rows:", len(df), "| Companies:", df["ticker"].nunique())
df.head()

df["fiscal_period_end"] = pd.to_datetime(df["fiscal_period_end"], errors="coerce")
full_history_df["fiscal_period_end"] = pd.to_datetime(
    full_history_df["fiscal_period_end"], errors="coerce"
)

numeric_columns = [
    "gross_profit", "total_assets", "stockholders_equity",
    "ordinary_shares_number", "fiscal_period_end_close",
]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

duplicate_keys = full_history_df.duplicated(
    ["ticker", "fiscal_period_end"], keep=False
)
if duplicate_keys.any():
    display(full_history_df.loc[duplicate_keys].sort_values(["ticker", "fiscal_period_end"]))
    raise ValueError("Duplicate firm-period records require investigation.")

df = (
    df.sort_values(["ticker", "fiscal_period_end"], ascending=[True, False])
      .drop_duplicates("ticker", keep="first")
      .copy()
)

df["market_equity"] = df["ordinary_shares_number"] * df["fiscal_period_end_close"]

columns_to_keep = [
    "ticker", "company_name", "sector", "industry",
    "fiscal_period_end", "fiscal_year", "gross_profit", "total_assets",
    "stockholders_equity", "ordinary_shares_number",
    "fiscal_period_end_close", "market_equity",
]
df = df[columns_to_keep]

assert df["ticker"].nunique() == len(df)
assert len(df) == EXPECTED_UNIVERSE_COUNT

print("Companies:", df["ticker"].nunique())
df[["ticker", "fiscal_period_end", "sector"]].head()

Data loaded from: https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv
Rows: 248 | Companies: 50
Companies: 50


,ticker,fiscal_period_end,sector
0,AAPL,2025-09-30,Technology
5,ABBV,2025-12-31,Healthcare
10,ABT,2025-12-31,Healthcare
15,ADBE,2025-11-30,Technology
20,AEP,2025-12-31,Utilities


In [16]:
import numpy as np
import pandas as pd

# The STA calculation already added 'net_income' and 'operating_cash_flow' to df.
# The SNOA component columns are also expected to be present in 'df' from previous successful merges.

# Define the list of columns required for SNOA calculation
required_cols_snoa_component = [
    'cash_and_cash_equivalents',
    'current_debt',
    'long_term_debt',
    'minority_interest',
    'preferred_stock',
    'common_stock_equity'
]

# Ensure 'net_income', 'operating_cash_flow', and 'total_assets' are numeric.
# These columns are expected to be present in 'df' from previous steps.
df['net_income'] = pd.to_numeric(df['net_income'], errors='coerce')
df['operating_cash_flow'] = pd.to_numeric(df['operating_cash_flow'], errors='coerce')
df['total_assets'] = pd.to_numeric(df['total_assets'], errors='coerce')


# --- STA Calculation ---
df['STA'] = (df['net_income'] - df['operating_cash_flow']) / df['total_assets'].replace(0, np.nan)


# --- SNOA Calculation ---
# Convert SNOA component columns to numeric and fill NaN values with 0 as requested
for col in required_cols_snoa_component:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce') # Ensure numeric before filling NaNs
        df[col] = df[col].fillna(0) # Fill NaN values with 0 as requested
    else:
        print(f"Warning: Column '{col}' not found in DataFrame for SNOA calculation. Assuming it was not merged successfully or is not in original full_history_df.")


# Calculate Operating Assets
df['operating_assets'] = df['total_assets'] - df['cash_and_cash_equivalents']

# Calculate Operating Liabilities
df['operating_liabilities'] = (
    df['total_assets']
    - df['current_debt']
    - df['long_term_debt']
    - df['minority_interest']
    - df['preferred_stock']
    - df['common_stock_equity']
)

# Calculate SNOA. Handle potential division by zero.
df['SNOA'] = (df['operating_assets'] - df['operating_liabilities']) / df['total_assets'].replace(0, np.nan)

print("Short-Term Accruals (STA) and Sum of Net Operating Assets (SNOA) have been calculated and added to the DataFrame.")
print("\nHere's a preview of the DataFrame with the new STA and SNOA columns:")
display(df[[
    'ticker', 'company_name', 'total_assets', 'net_income', 'operating_cash_flow', 'STA',
    'cash_and_cash_equivalents', 'operating_assets', 'current_debt', 'long_term_debt',
    'minority_interest', 'preferred_stock', 'common_stock_equity', 'operating_liabilities', 'SNOA'
]].head())

Short-Term Accruals (STA) and Sum of Net Operating Assets (SNOA) have been calculated and added to the DataFrame.

Here's a preview of the DataFrame with the new STA and SNOA columns:


,ticker,company_name,total_assets,net_income,operating_cash_flow,STA,cash_and_cash_equivalents,operating_assets,current_debt,long_term_debt,minority_interest,preferred_stock,common_stock_equity,operating_liabilities,SNOA
0,AAPL,Apple Inc.,3.592410e+11,1.120100e+11,1.114820e+11,0.001470,3.593400e+10,3.233070e+11,2.032900e+10,7.832800e+10,0.000000e+00,0.0,7.373300e+10,1.868510e+11,0.379845
1,ABBV,AbbVie Inc.,1.339600e+11,4.226000e+09,1.903000e+10,-0.110511,5.229000e+09,1.287310e+11,8.555000e+09,5.894100e+10,4.200000e+07,0.0,-3.270000e+09,6.969200e+10,0.440721
2,ABT,Abbott Laboratories,8.671300e+10,6.524000e+09,9.566000e+09,-0.035081,8.522000e+09,7.819100e+10,3.033000e+09,9.896000e+09,6.410000e+08,0.0,5.213000e+10,2.101300e+10,0.659394
3,ADBE,Adobe Inc.,2.949600e+10,7.130000e+09,1.003100e+10,-0.098352,5.431000e+09,2.406500e+10,0.000000e+00,6.210000e+09,0.000000e+00,0.0,1.162300e+10,1.166300e+10,0.420464
4,AEP,"American Electric Power Company, Inc.",1.144600e+11,3.580000e+09,6.944000e+09,-0.029390,1.970000e+08,1.142630e+11,4.702000e+09,4.412800e+10,1.080000e+09,0.0,3.113800e+10,3.341200e+10,0.706369


In [17]:
import numpy as np

# Rank STA: lowest value is best (rank 1)
df['STA_rank'] = df['STA'].rank(ascending=True, method='min')

# Rank SNOA: lowest value is best (rank 1)
df['SNOA_rank'] = df['SNOA'].rank(ascending=True, method='min')

print("Companies ranked by STA (1=best, lowest STA):")
display(df[['company_name', 'STA', 'STA_rank']].sort_values(by='STA_rank').head())
display(df[['company_name', 'STA', 'STA_rank']].sort_values(by='STA_rank').tail())

print("\nCompanies ranked by SNOA (1=best, lowest SNOA):")
display(df[['company_name', 'SNOA', 'SNOA_rank']].sort_values(by='SNOA_rank').head())
display(df[['company_name', 'SNOA', 'SNOA_rank']].sort_values(by='SNOA_rank').tail())

Companies ranked by STA (1=best, lowest STA):


,company_name,STA,STA_rank
1,AbbVie Inc.,-0.110511,1.0
3,Adobe Inc.,-0.098352,2.0
16,"EOG Resources, Inc.",-0.097763,3.0
9,ConocoPhillips,-0.096835,4.0
39,Starbucks Corporation,-0.090291,5.0


,company_name,STA,STA_rank
29,Morgan Stanley,0.024467,46.0
18,"The Goldman Sachs Group, Inc.",0.034449,47.0
23,JPMorgan Chase & Co.,0.046290,48.0
24,The Coca-Cola Company,0.054371,49.0
33,NVIDIA Corporation,0.083891,50.0



Companies ranked by SNOA (1=best, lowest SNOA):


,company_name,SNOA,SNOA_rank
47,Wells Fargo & Company,0.093950,1.0
23,JPMorgan Chase & Co.,0.117309,2.0
6,Bank of America Corporation,0.125985,3.0
39,Starbucks Corporation,0.148840,4.0
18,"The Goldman Sachs Group, Inc.",0.190510,5.0


,company_name,SNOA,SNOA_rank
15,Duke Energy Corporation,0.728583,46.0
31,"NextEra Energy, Inc.",0.748802,47.0
33,NVIDIA Corporation,0.750260,48.0
43,Thermo Fisher Scientific Inc.,0.752825,49.0
14,Deere & Company,0.770105,50.0


In [18]:
# Calculate cumulative rank by summing STA_rank and SNOA_rank
df['cumulative_rank_score'] = df['STA_rank'] + df['SNOA_rank']

# Rank companies based on the cumulative score (lowest score is best)
df['overall_rank'] = df['cumulative_rank_score'].rank(ascending=True, method='min')

print("Companies ranked by combined STA and SNOA (1=best, lowest cumulative rank):")
display(df[['company_name', 'STA_rank', 'SNOA_rank', 'cumulative_rank_score', 'overall_rank']].sort_values(by='overall_rank').head(10))

print("\nBottom 10 companies by combined STA and SNOA (highest cumulative rank is worst):")
display(df[['company_name', 'STA_rank', 'SNOA_rank', 'cumulative_rank_score', 'overall_rank']].sort_values(by='overall_rank').tail(10))

Companies ranked by combined STA and SNOA (1=best, lowest cumulative rank):


,company_name,STA_rank,SNOA_rank,cumulative_rank_score,overall_rank
39,Starbucks Corporation,5.0,4.0,9.0,1.0
1,AbbVie Inc.,1.0,14.0,15.0,2.0
3,Adobe Inc.,2.0,13.0,15.0,2.0
10,Costco Wholesale Corporation,8.0,8.0,16.0,4.0
5,"Amazon.com, Inc.",6.0,16.0,22.0,5.0
42,"The TJX Companies, Inc.",18.0,6.0,24.0,6.0
48,Walmart Inc.,7.0,17.0,24.0,6.0
27,"Altria Group, Inc.",10.0,20.0,30.0,8.0
46,"United Parcel Service, Inc.",17.0,15.0,32.0,9.0
25,"Lowe's Companies, Inc.",13.0,22.0,35.0,10.0



Bottom 10 companies by combined STA and SNOA (highest cumulative rank is worst):


,company_name,STA_rank,SNOA_rank,cumulative_rank_score,overall_rank
19,"The Home Depot, Inc.",33.0,34.0,67.0,41.0
22,Johnson & Johnson,43.0,24.0,67.0,41.0
4,"American Electric Power Company, Inc.",25.0,44.0,69.0,43.0
31,"NextEra Energy, Inc.",28.0,47.0,75.0,44.0
28,"Merck & Co., Inc.",44.0,33.0,77.0,45.0
36,Pfizer Inc.,34.0,45.0,79.0,46.0
14,Deere & Company,30.0,50.0,80.0,47.0
43,Thermo Fisher Scientific Inc.,36.0,49.0,85.0,48.0
24,The Coca-Cola Company,49.0,40.0,89.0,49.0
33,NVIDIA Corporation,50.0,48.0,98.0,50.0
